In [1]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

root = str(Path.cwd().parent)

if root not in sys.path:
    sys.path.append(root)

from src.step2_transformation import DataTransformer
from src.step3a_bwm_model import BWMCalculator
from src.step3b_hierarchical_bwm import HierarchicalAggregator
from src.step4_saw_aggregation import SAWCalculator

In [2]:
# =========================================================
# PHASE 2: NORMALIZATION (The Data Engineering)
# =========================================================
print("🔄 Starting Pipeline...")
transformer = DataTransformer(config_name='step1_features_config.yaml', max_budget=105000)
df_normalized = transformer.execute_pipeline()

🔄 Starting Pipeline...
✅ YAML Config loaded successfully.

🚀 Starting ETL pipeline for file: data/car_database.csv
Filter 1: Applying Hard Constraints dynamically...
  -> Applied constraint: cost <= 105000
 -> Cars removed: 17 | Remaining: 14
Filter 2: Removing uninformative columns...
  -> Auto-dropped (Zero Variance): ['hill_start_assist', 'handling', 'leather_steering_wheel', 'traction_control', 'power_door_locks', 'driver_seat_height_adjustment', 'stability_control', 'power_steering', 'transmission', 'abs', 'leather_seats']
Transformation 1: Applying Conditional Logic dynamically...
  -> Applied FILL_NULL (1.0) for: fog_lights
  -> Applied OR_GATE substitution for: rear_view_camera
  -> Applied OR_GATE substitution for: rear_parking_sensors
Transformation 2: Applying Utility Mapping...
Transformation 3: Binarizing booleans...
Transformation 4: Continuous Normalization (Min-Max)...
✅ Pipeline Finished! Mathematical matrix generated successfully.



In [3]:
# =========================================================
# PHASE 3: MACRO BWM (The Main Decision Profile)
# =========================================================
macro_categories = ["cost", "safety", "performance", "comfort", "aesthetics"]

best_cat = "cost"
worst_cat = "aesthetics"

# The user's macro preferences (1 to 9)
bo_macro = {"safety": 2, "performance": 4, "comfort": 6, "aesthetics": 8}
ow_macro = {"cost": 8, "safety": 4, "performance": 2, "comfort": 1}

bwm = BWMCalculator(macro_categories)
macro_weights, cr = bwm.calculate_weights(best_cat, worst_cat, bo_macro, ow_macro)

In [4]:
# =========================================================
# PHASE 4: HIERARCHICAL AGGREGATION (The Specifics)
# =========================================================
# The user opened the "Advanced" tab ONLY in the Safety category
custom_micro = {
    'safety': {
        'rear_view_camera': 0.50,      
        'airbag': 0.40,                
        'rear_parking_sensors': 0.05,
        'tire_pressure_sensor': 0.05
    }
}

aggregator = HierarchicalAggregator(config_name='step1_features_config.yaml')
absolute_weights = aggregator.generate_flat_weights(macro_weights, custom_micro)

✅ Hierarchical Groups loaded: ['cost', 'safety', 'performance', 'comfort', 'aesthetics']
📊 Absolute Weights generated for 28 features (Sum: 1.0002)


In [5]:
# =========================================================
# PHASE 5: SAW AGGREGATION (The Final Decision)
# =========================================================
saw = SAWCalculator()
df_final_ranking = saw.calculate_ranking(df_normalized, absolute_weights)

Aggregation: Calculating Final SAW Scores...
✅ Ranking generated successfully!


In [6]:
# =========================================================
# VISUALIZATION: THE ULTIMATE REPORT
# =========================================================
print(f"\n📊 BWM Consistency Ratio: {cr}")
if cr < 0.1:
    print("✅ Perfect mathematical consistency.")
else:
    print("⚠️ Consistency acceptable but has minor deviations.")

print("\n🏆 THE FINAL TOP 5 RECOMMENDED CARS:")

# Select the most important columns to show (IDs + Final Score + The particularities the user cared about)
columns_to_show = ['car', 'version', 'Final_Score', 'cost', 'rear_view_camera', 'airbag']

# Filter only the columns that actually exist in the dataframe to avoid KeyError
valid_columns = [col for col in columns_to_show if col in df_final_ranking.columns]

df_top5 = df_final_ranking[valid_columns].head(5)

# Render a beautiful DataFrame highlighting the final score
display(df_top5)


📊 BWM Consistency Ratio: 0.0035
✅ Perfect mathematical consistency.

🏆 THE FINAL TOP 5 RECOMMENDED CARS:


,car,version,Final_Score,cost,rear_view_camera,airbag
0,Hyundai Hb20 2026,1.0 12V FLEX LIMITED MANUAL,0.696636,0.212121,1.0,1.0
1,Peugeot 208 2026,1.0 FIREFLY FLEX STYLE MANUAL,0.609656,0.412121,1.0,0.8
2,Citroën C3 2026,1.0 FIREFLY FLEX FEEL MANUAL,0.564635,0.878788,1.0,0.5
3,Chevrolet Onix 2026,1.0 FLEX MANUAL + Sensor,0.561025,0.015879,1.0,1.0
4,Hyundai Hb20 2026,1.0 12V FLEX COMFORT MANUAL,0.555668,0.460606,0.0,1.0
